In [1]:
import cv2 as cv
import numpy as np

(Podla mna tomu morfologia iba skodi na to sa musime kuknut este ale
pri 3x3 matici to vezme kus intenzity peaku... a 2x2 je useless imo)

In [2]:
# Load image
img = cv.imread("test.png", cv.IMREAD_COLOR)
assert img is not None

# ==========================================================================================
# Preprocessing: PNG → BGR → grayscale → gaussian blur → threshold → processing
# ==========================================================================================

# Apply grayscale to image
gray_image = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

# Apply gaussian blur to image
gauss_image = cv.GaussianBlur(gray_image, (3, 3), 0)

# Apply threshold to image
_, thresh_image = cv.threshold(gauss_image, 0, 255, cv.THRESH_BINARY_INV + cv.THRESH_OTSU)

# Show images
cv.imshow("With grayscale", gray_image)
cv.imshow("With gaussian blur", gauss_image)
cv.imshow("With threshold", thresh_image)
cv.waitKey(0)
cv.destroyAllWindows()



QFontDatabase: Cannot find font directory /home/richard/Desktop/NMR-Spectra_Analysis-Model/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/richard/Desktop/NMR-Spectra_Analysis-Model/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/richard/Desktop/NMR-Spectra_Analysis-Model/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/richard/Desktop/NMR-Spectra_Analysis-Model/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fon

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Importy
# ─────────────────────────────────────────────────────────────────────────────
import os
import ast
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split

# ─────────────────────────────────────────────────────────────────────────────
# Konfigurácia
# ─────────────────────────────────────────────────────────────────────────────
DATASET_DIR    = "../nmr_dataset"
INDEX_CSV      = os.path.join(DATASET_DIR, "index.csv")
LABEL_MAP_JSON = os.path.join(DATASET_DIR, "label_map.json")

TEST_SIZE      = 0.2
RANDOM_STATE   = 42
BATCH_SIZE     = 32
EPOCHS         = 50
LEARNING_RATE  = 0.001
WEIGHT_DECAY   = 1e-4
MODEL_TYPE     = "cnn"    # "cnn" alebo "mlp"
INPUT_LEN      = 1024     # dĺžka vektora z generátora

# ─────────────────────────────────────────────────────────────────────────────
# Načítanie dát
# ─────────────────────────────────────────────────────────────────────────────

with open(LABEL_MAP_JSON, "r") as f:
    label_map = json.load(f)

# Invertovaná mapa: číslo → názov zlúčeniny (na výpis výsledkov)
inv_label_map = {v: k for k, v in label_map.items()}
num_classes   = len(label_map)
print(f"Počet tried: {num_classes}")

index_df = pd.read_csv(INDEX_CSV)
print(f"Celkovo vzoriek v indexe: {len(index_df)}")
print(f"  Čisté:  {(~index_df['is_mixture']).sum()}")
print(f"  Zmesi:  {index_df['is_mixture'].sum()}")


def label_to_multihot(label_val, is_mixture: bool, num_classes: int) -> np.ndarray:
    """
    Prevedie label na multi-hot vektor.

    Pre čisté zlúčeniny:
        label = 5  →  [0, 0, 0, 0, 0, 1, 0, ...]

    Pre zmesi:
        label = "[0, 5]"  →  [1, 0, 0, 0, 0, 1, 0, ...]

    Multi-hot znamená že môže byť zapnutých viacero bitov naraz —
    každý bit hovorí "táto zlúčenina je/nie je prítomná".
    """
    vec = np.zeros(num_classes, dtype=np.float32)

    if is_mixture:
        # Label je string napr. "[0, 5]" — parsujeme ho na zoznam čísel
        indices = ast.literal_eval(str(label_val))
        for idx in indices:
            vec[idx] = 1.0
    else:
        vec[int(label_val)] = 1.0

    return vec


def load_dataset(index_df, num_classes):
    """
    Načíta všetky .npy vektory a prevedie labely na multi-hot vektory.
    """
    X = []
    Y = []

    for _, row in index_df.iterrows():
        npy_path = row["npy_path"]

        # Preskočíme riadky bez súboru
        if not npy_path or not os.path.exists(str(npy_path)):
            continue

        vec   = np.load(npy_path).astype(np.float32)
        label = label_to_multihot(row["label"], row["is_mixture"], num_classes)

        X.append(vec)
        Y.append(label)

    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)


print("\nNačítavam vektory...")
X, Y = load_dataset(index_df, num_classes)
print(f"X tvar: {X.shape}")   # (n_vzoriek, 1024)
print(f"Y tvar: {Y.shape}")   # (n_vzoriek, num_classes)

# ─────────────────────────────────────────────────────────────────────────────
# Train/test split
# ─────────────────────────────────────────────────────────────────────────────

# Pre zmesi nemôžeme použiť stratify (viacero labelov) —
# rozdelíme jednoducho náhodne
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
)
print(f"Train: {X_train.shape[0]} vzoriek")
print(f"Test:  {X_test.shape[0]} vzoriek")

# CNN potrebuje tvar (batch, steps, channels)
if MODEL_TYPE == "cnn":
    X_train = X_train[..., np.newaxis]   # (n, 1024, 1)
    X_test  = X_test[..., np.newaxis]

# ─────────────────────────────────────────────────────────────────────────────
# Definícia modelu
# ─────────────────────────────────────────────────────────────────────────────

if MODEL_TYPE == "cnn":
    model = models.Sequential([
        # Prvý blok — zachytáva hrubé vzory (celkový tvar spektra)
        layers.Conv1D(32, kernel_size=7, activation="relu", padding="same",
                      input_shape=(INPUT_LEN, 1)),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),

        # Druhý blok — zachytáva stredné vzory (skupiny píkov)
        layers.Conv1D(64, kernel_size=5, activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),

        # Tretí blok — zachytáva jemné vzory (individuálne píky)
        layers.Conv1D(128, kernel_size=3, activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),

        # Štvrtý blok — hlbšie features
        layers.Conv1D(256, kernel_size=3, activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),

        layers.Dropout(0.4),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),

        # ── Kľúčová zmena oproti starému kódu ──────────────────────────────
        # sigmoid namiesto softmax — každá trieda má vlastnú pravdepodobnosť
        # softmax: súčet všetkých = 1.0 (iba jeden víťaz)
        # sigmoid: každá trieda nezávisle 0.0–1.0 (môže byť viac víťazov)
        layers.Dense(num_classes, activation="sigmoid"),
    ])

elif MODEL_TYPE == "mlp":
    model = models.Sequential([
        layers.Flatten(input_shape=(INPUT_LEN,)),
        layers.Dense(512, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),

        # sigmoid pre multi-label
        layers.Dense(num_classes, activation="sigmoid"),
    ])

# ── Kľúčová zmena v loss funkcii ────────────────────────────────────────────
# BinaryCrossentropy namiesto CategoricalCrossentropy
# Binary = každá trieda je samostatná binárna otázka "je prítomná alebo nie?"
# Categorical = iba jeden správny výsledok z mnohých
model.compile(
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate = LEARNING_RATE,
        weight_decay  = WEIGHT_DECAY,
    ),
    loss    = tf.keras.losses.BinaryCrossentropy(),
    metrics = [
        tf.keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.5),
        tf.keras.metrics.AUC(name="auc", multi_label=True),
    ],
)

model.summary()

# ─────────────────────────────────────────────────────────────────────────────
# Tréning
# ─────────────────────────────────────────────────────────────────────────────

early_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6
)

history = model.fit(
    X_train, Y_train,
    batch_size       = BATCH_SIZE,
    epochs           = EPOCHS,
    validation_split = 0.2,
    callbacks        = [early_stop, reduce_lr],
    verbose          = 1,
)

# ─────────────────────────────────────────────────────────────────────────────
# Evaluácia
# ─────────────────────────────────────────────────────────────────────────────

test_loss, test_acc, test_auc = model.evaluate(X_test, Y_test, verbose=0)
print(f"\nTest loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test AUC:      {test_auc:.4f}")

# Plot tréningovej histórie
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"],     label="Train acc")
axes[0].plot(history.history["val_accuracy"], label="Val acc")
axes[0].set_title("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"],     label="Train loss")
axes[1].plot(history.history["val_loss"], label="Val loss")
axes[1].set_title("Loss")
axes[1].legend()

plt.tight_layout()
plt.savefig("training_history.png")
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# Uloženie modelu
# ─────────────────────────────────────────────────────────────────────────────

# Ulož model
model.save(os.path.join(os.path.dirname(INDEX_CSV), "..", "nmr_classifier.h5"))
print("Model uložený ako 'nmr_classifier.h5'")

# Ulož label mapu — potrebná na inferenciu
with open("label_map.json", "w") as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)
print("Label mapa uložená ako 'label_map.json'")

# Ulož aj konfiguráciu modelu — užitočné na neskoršie načítanie
config = {
    "model_type":  MODEL_TYPE,
    "input_len":   INPUT_LEN,
    "num_classes": num_classes,
    "threshold":   0.5,
}
with open("model_config.json", "w") as f:
    json.dump(config, f, indent=2)
print("Konfigurácia uložená ako 'model_config.json'")


# ─────────────────────────────────────────────────────────────────────────────
# Inferencia
# ─────────────────────────────────────────────────────────────────────────────

def predict_from_vector(model, vector: np.ndarray, inv_label_map: dict,
                        threshold: float = 0.5, model_type: str = "cnn"):
    """
    Predikuje zlúčeniny priamo z numpy vektora — bez potreby súboru.

    Parametre:
        model         : natrénovaný model
        vector        : 1D numpy array (1024 hodnôt)
        inv_label_map : slovník {index → názov zlúčeniny}
        threshold     : minimálna pravdepodobnosť (default 0.5)
        model_type    : "cnn" alebo "mlp"
    """
    if model_type == "cnn":
        vec_input = vector.reshape(1, -1, 1)   # (1, 1024, 1)
    else:
        vec_input = vector.reshape(1, -1)       # (1, 1024)

    probs = model.predict(vec_input, verbose=0)[0]

    detected = [
        (inv_label_map[i], float(probs[i]))
        for i in range(len(probs))
        if probs[i] >= threshold
    ]
    detected.sort(key=lambda x: -x[1])
    return detected


def predict_from_file(model, npy_path: str, inv_label_map: dict,
                      threshold: float = 0.5, model_type: str = "cnn"):
    """
    Predikuje zlúčeniny z .npy súboru na disku.
    """
    vector = np.load(npy_path).astype(np.float32)
    return predict_from_vector(model, vector, inv_label_map, threshold, model_type)


test_idx = 0

# (odstráň channel dimenziu ak CNN)
if MODEL_TYPE == "cnn":
    sample_vector = X_test[test_idx, :, 0]   # (1024,)
else:
    sample_vector = X_test[test_idx]          # (1024,)

# Predikcia
result = predict_from_vector(
    model         = model,
    vector        = sample_vector,
    inv_label_map = inv_label_map,
    threshold     = 0.5,
    model_type    = MODEL_TYPE,
)

print("Predikcia pre testovaciu vzorku:")
for name, prob in result:
    print(f"  {name:25s}: {prob:.3f} ({prob*100:.1f}%)")

print("\nSkutočný label:", [inv_label_map[i] for i in range(num_classes)
                            if Y_test[test_idx][i] > 0.5])

# ── Otestuj viacero vzoriek naraz ────────────────────────────────────────────
print("\n=== Ukážka 5 náhodných predikcií ===")
for idx in range(5):
    if MODEL_TYPE == "cnn":
        vec = X_test[idx, :, 0]
    else:
        vec = X_test[idx]

    pred    = predict_from_vector(model, vec, inv_label_map, threshold=0.5, model_type=MODEL_TYPE)
    true    = [inv_label_map[i] for i in range(num_classes) if Y_test[idx][i] > 0.5]
    correct = set([p[0] for p in pred]) == set(true)

    print(f"  [{idx}] Skutočné: {true}")
    print(f"       Predikcia: {[p[0] for p in pred]}")
    print(f"       {'✓ Správne' if correct else '✗ Chyba'}")


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# Invert map: index -> compound name
inv_label_map = {v: k for k, v in label_map.items()}
print("Label map:", inv_label_map)

predicted_X = model.predict(X)
y_onehot = tf.keras.utils.to_categorical(y, num_classes=42)
my_rmse = rmse(y_onehot, predicted_X)
print(f"RMSE: (rmse = {my_rmse:.5f})")